In [3]:
import sys
import os
import time
sys.path.append(os.path.abspath("../src"))
import pandas as pd
from data_loader import *

In [3]:
all_leagues = get_all_leagues()

In [4]:
tournaments = [
    "Slam IV",
    "FISSURE PLAYGROUND 2",
    "PGL Wallachia 2025 Season 6",
    "Slam V",
    "DreamLeague Season 27",
    "BLAST Slam VI",
    "DreamLeague Season 28",
]

In [ ]:
leagues_ids = []

for name in tournaments:
    result = find_main_event_by_name(name, all_leagues)
    print(result)
    
    if not result.empty:
        leagues_ids.extend(result["leagueid"].tolist())
leagues_ids = list(set(leagues_ids))
print("Final League IDs:", leagues_ids)

In [ ]:
test_id = leagues_ids[0]
df = get_league_matches(test_id)
print(df.shape)

In [ ]:
all_matches = []

for lid in leagues_ids:
    df = get_league_matches(lid)
    print(f"League {lid}: {df.shape}")
    all_matches.append(df)
    time.sleep(1)
    
match_df = pd.concat(all_matches, ignore_index=True)
print("Total Matches:", match_df.shape)    

In [ ]:
match_df.to_csv("../data/raw/raw_matches.csv", index=False)

In [10]:
df = pd.read_csv("../data/processed/clean_matches.csv")

In [10]:
teams =  pd.concat([
    df["radiant_team_id"],
    df["dire_team_id"]
]).dropna().unique()

print(len(teams))

35


In [11]:
team_map = {}

for team_id in teams:
    data = get_team_info(int(team_id))
    team_map[team_id] = data.get("name")

    time.sleep(1)

In [12]:
df["radiant_team_name"] = df["radiant_team_id"].map(team_map)
df["dire_team_name"] = df["dire_team_id"].map(team_map)

In [ ]:
team_df = pd.DataFrame(
    team_map.items(),
    columns=["team_id", "team_name"]
)

team_df.head()

In [14]:
team_df.to_csv("../data/processed/team_mapping.csv", index=False)

In [20]:
match_detail = []
for md in match_df["match_id"]:
    detail_df = get_match_detail(md)
    match_detail.append(detail_df)
    time.sleep(1)
    
match_detail_df = pd.DataFrame(match_detail)

In [ ]:
print(match_detail_df.shape)
print(match_detail_df.columns.tolist())

In [24]:
match_detail_df.to_csv("../data/raw/match_detail.csv")

In [ ]:
all_picks = []
for i, match_id in enumerate(df["match_id"]):
    try:
        picks = get_picks_bans(match_id)
        all_picks.extend(picks)
    except Exception as e:
        print(f"Error match {match_id}: {e}")
        continue
    
    
    if i % 50 == 0:
        print(f"Progress: {i}/{len(df)}")
        pd.DataFrame(all_picks).to_csv("../data/raw/picks_bans.csv", index=False)
    
    time.sleep(2)

# Final save
picks_df = pd.DataFrame(all_picks)
picks_df.to_csv("../data/raw/picks_bans.csv", index=False)
print(f"Done! Total picks: {len(picks_df)}")

In [13]:
picks_df = pd.read_csv("../data/raw/picks_bans.csv")

In [ ]:
print(f"Total picks fetched: {len(all_picks)}")
print(f"Unique matches: {pd.DataFrame(all_picks)['match_id'].nunique()}")

In [ ]:
all_match_ids = set(df["match_id"])

success_match_ids = set(picks_df["match_id"])

failed_matches = list(all_match_ids - success_match_ids)

print(len(failed_matches))

In [ ]:
retry_picks = []

for i, match_id in enumerate(failed_matches):
    try:
        picks = get_picks_bans(match_id)
        retry_picks.extend(picks)
    except Exception as e:
        print(f"Still failed: {match_id}")
    
    if i % 10 == 0 and i != 0:
        print(f"Retry progress: {i}/{len(failed_matches)}")

    time.sleep(3)


In [18]:
all_picks = picks_df.to_dict("records")
all_picks.extend(retry_picks)

In [ ]:
final_df = pd.DataFrame(all_picks)
final_df.to_csv("../data/raw/picks_bans_final.csv", index=False)

print(f"Final total picks: {len(final_df)}")
print(f"Final unique matches: {final_df['match_id'].nunique()}")

In [ ]:
final_df["match_id"].nunique()